# 🧪 Vedaz Qwen Fine-Tuning (QLoRA)

Fine-tune **Qwen 2.5** models using **QLoRA** for the Vedaz AI Vedic Astrologer.

**Requirements:** Google Colab with T4 GPU (free tier works!)

---

## Pipeline
1. ✅ GPU Check & Dependencies
2. 📤 Upload & Process Data
3. 🏋️ Train (QLoRA + SFTTrainer)
4. 📊 Evaluate
5. 💬 Test (Interactive Inference)
6. 🚀 Merge & Export Model

---
## 1. GPU Check & Install Dependencies

In [ ]:
# Check GPU availability
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("NO GPU DETECTED!")
    print("Go to Runtime > Change runtime type > Select T4 GPU")

In [ ]:
%%capture
# Install dependencies (takes ~2-3 min)
!pip install -q torch transformers datasets accelerate peft bitsandbytes trl tensorboard pyyaml

In [ ]:
# Verify installations
import transformers, datasets, peft, trl, bitsandbytes, torch
print(f"Transformers: {transformers.__version__}")
print(f"Datasets:     {datasets.__version__}")
print(f"PEFT:         {peft.__version__}")
print(f"TRL:          {trl.__version__}")
print(f"BnB:          {bitsandbytes.__version__}")
print(f"PyTorch:      {torch.__version__}")
print(f"CUDA:         {torch.version.cuda}")

---
## 2. Upload & Process Data

Upload your `Chat Data for assessment of applicants.json` file when prompted.

In [ ]:
import os
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data", exist_ok=True)
os.makedirs("outputs/checkpoints", exist_ok=True)
os.makedirs("outputs/merged_model", exist_ok=True)
os.makedirs("logs", exist_ok=True)

In [ ]:
# Upload your data file
from google.colab import files
uploaded = files.upload()

# Move uploaded file to data/raw/
for filename in uploaded:
    os.rename(filename, f"data/raw/{filename}")
    print(f"Moved {filename} -> data/raw/{filename}")

In [ ]:
# ============================================
# Data Processing
# ============================================
import json
import random


def load_mixed_json(filepath):
    """Load mixed-format JSON (single-line JSONL + multi-line objects)."""
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read().strip()

    decoder = json.JSONDecoder()
    objects = []
    idx = 0

    while idx < len(content):
        while idx < len(content) and content[idx] in " \t\r\n,":
            idx += 1
        if idx >= len(content):
            break
        try:
            obj, end = decoder.raw_decode(content[idx:])
            objects.append(obj)
            idx += end
        except json.JSONDecodeError:
            idx += 1
    return objects


def normalize_conversation(obj):
    """Keep only 'messages' key, validate structure."""
    if "messages" not in obj:
        return None
    messages = obj["messages"]
    if not isinstance(messages, list) or len(messages) < 2:
        return None

    cleaned = []
    for msg in messages:
        if not isinstance(msg, dict) or "role" not in msg or "content" not in msg:
            return None
        if msg["role"] not in ("system", "user", "assistant"):
            return None
        if not msg["content"].strip():
            return None
        cleaned.append({"role": msg["role"], "content": msg["content"].strip()})

    roles = {m["role"] for m in cleaned}
    if "user" not in roles or "assistant" not in roles:
        return None
    return {"messages": cleaned}


# Process all files in data/raw/
all_data = []
for file in sorted(os.listdir("data/raw")):
    if file.endswith((".json", ".jsonl")):
        filepath = f"data/raw/{file}"
        print(f"\nLoading: {file}")
        raw = load_mixed_json(filepath)
        print(f"  Raw objects: {len(raw)}")

        valid = [normalize_conversation(obj) for obj in raw]
        valid = [v for v in valid if v is not None]
        print(f"  Valid conversations: {len(valid)}")
        all_data.extend(valid)

# Stats
total_turns = sum(len(d["messages"]) for d in all_data)
user_msgs = sum(1 for d in all_data for m in d["messages"] if m["role"] == "user")
asst_msgs = sum(1 for d in all_data for m in d["messages"] if m["role"] == "assistant")
multi_turn = sum(1 for d in all_data if sum(1 for m in d["messages"] if m["role"] == "user") > 1)
total_chars = sum(len(m["content"]) for d in all_data for m in d["messages"])

print(f"\n{'='*50}")
print(f"  Dataset Statistics")
print(f"{'='*50}")
print(f"  Conversations:       {len(all_data)}")
print(f"  Total turns:         {total_turns}")
print(f"  User messages:       {user_msgs}")
print(f"  Assistant messages:  {asst_msgs}")
print(f"  Multi-turn:          {multi_turn}")
print(f"  Total characters:    {total_chars:,}")
print(f"  Est. tokens:         ~{total_chars // 3:,}")

In [ ]:
# Train/Val Split
VAL_RATIO = 0.1
SEED = 42

random.seed(SEED)
shuffled = all_data.copy()
random.shuffle(shuffled)
split_idx = int(len(shuffled) * (1 - VAL_RATIO))
train_data = shuffled[:split_idx]
val_data = shuffled[split_idx:]

# Save as JSONL
def save_jsonl(data, filepath):
    with open(filepath, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

save_jsonl(train_data, "data/train.jsonl")
save_jsonl(val_data, "data/val.jsonl")

print(f"Train: {len(train_data)} conversations -> data/train.jsonl")
print(f"Val:   {len(val_data)} conversations -> data/val.jsonl")

---
## 3. Training Configuration

Adjust these parameters based on your GPU and dataset size.

In [ ]:
# ============================================
# Configuration
# ============================================

# --- Model ---
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"  # Change to 3B or 1.5B for smaller GPUs
TORCH_DTYPE = torch.bfloat16

# --- LoRA ---
LORA_R = 64
LORA_ALPHA = 128
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# --- Quantization (QLoRA) ---
USE_4BIT = True
BNB_4BIT_COMPUTE_DTYPE = torch.bfloat16
BNB_4BIT_QUANT_TYPE = "nf4"
BNB_4BIT_USE_DOUBLE_QUANT = True

# --- Training ---
OUTPUT_DIR = "outputs/checkpoints"
NUM_EPOCHS = 3
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 8      # Effective batch size = 2 * 8 = 16
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.03
LR_SCHEDULER = "cosine"
MAX_SEQ_LENGTH = 2048
LOGGING_STEPS = 5
SAVE_STEPS = 50
EVAL_STEPS = 50
MAX_GRAD_NORM = 0.3
OPTIM = "paged_adamw_8bit"
SEED = 42

print(f"Model: {MODEL_NAME}")
print(f"LoRA: r={LORA_R}, alpha={LORA_ALPHA}")
print(f"Quantization: {'4-bit NF4' if USE_4BIT else 'None'}")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"Max sequence length: {MAX_SEQ_LENGTH}")

---
## 4. Load Model & Tokenizer

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Quantization config
bnb_config = None
if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=BNB_4BIT_COMPUTE_DTYPE,
        bnb_4bit_quant_type=BNB_4BIT_QUANT_TYPE,
        bnb_4bit_use_double_quant=BNB_4BIT_USE_DOUBLE_QUANT,
    )

# Load tokenizer
print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Load model
print(f"Loading model: {MODEL_NAME}")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=TORCH_DTYPE,
    attn_implementation="sdpa",
    device_map="auto",
    trust_remote_code=True,
)

if bnb_config:
    model = prepare_model_for_kbit_training(model)

model.config.use_cache = False
print(f"Model loaded on: {model.device}")

In [ ]:
# Apply LoRA
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    task_type="CAUSAL_LM",
    bias="none",
)

model = get_peft_model(model, lora_config)

trainable, total = model.get_nb_trainable_parameters()
print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

---
## 5. Prepare Dataset

In [ ]:
from datasets import load_dataset

# Load processed data
dataset = load_dataset("json", data_files={
    "train": "data/train.jsonl",
    "validation": "data/val.jsonl",
})

print(f"Train: {len(dataset['train'])} samples")
print(f"Val:   {len(dataset['validation'])} samples")

# Preview a sample
sample = dataset["train"][0]
print(f"\nSample conversation ({len(sample['messages'])} turns):")
for msg in sample["messages"]:
    role = msg["role"].upper()
    content = msg["content"][:80] + "..." if len(msg["content"]) > 80 else msg["content"]
    print(f"  [{role}] {content}")

In [ ]:
# Apply chat template to format messages
def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

formatted_train = dataset["train"].map(
    format_chat,
    remove_columns=dataset["train"].column_names,
)
formatted_val = dataset["validation"].map(
    format_chat,
    remove_columns=dataset["validation"].column_names,
)

print(f"Formatted train: {len(formatted_train)} samples")
print(f"\nSample formatted text (first 500 chars):")
print(formatted_train[0]["text"][:500])

---
## 6. Train!

In [ ]:
from trl import SFTTrainer, SFTConfig
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_output_dir = f"{OUTPUT_DIR}/run_{timestamp}"

training_args = SFTConfig(
    output_dir=run_output_dir,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=10,  # replaced warmup_ratio to avoid warning
    lr_scheduler_type=LR_SCHEDULER,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    eval_steps=EVAL_STEPS,
    eval_strategy="steps",
    save_total_limit=3,
    fp16=False,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim=OPTIM,
    max_grad_norm=MAX_GRAD_NORM,
    report_to="tensorboard",
    logging_dir=f"logs/run_{timestamp}",
    seed=SEED,
    max_length=MAX_SEQ_LENGTH,
    packing=False,
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=formatted_train,
    eval_dataset=formatted_val,
    processing_class=tokenizer,
)

print(f"Output: {run_output_dir}")
print(f"Starting training...")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Effective batch: {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"  LR: {LEARNING_RATE}")

In [ ]:
# Start training
train_result = trainer.train()

# Save final adapter
trainer.save_model()
tokenizer.save_pretrained(run_output_dir)

# Log metrics
metrics = train_result.metrics
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)
trainer.save_state()

print(f"\nTraining complete!")
print(f"Adapter saved to: {run_output_dir}")
for k, v in metrics.items():
    print(f"  {k}: {v}")

---
## 7. View Training Logs

In [ ]:
# Launch TensorBoard
%load_ext tensorboard
%tensorboard --logdir logs/

---
## 8. Evaluate

In [ ]:
import math

# Compute validation perplexity
model.eval()
total_loss = 0.0
total_tokens = 0

val_dataset_raw = load_dataset("json", data_files="data/val.jsonl", split="train")

for example in val_dataset_raw:
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH).to(model.device)

    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])

    total_loss += outputs.loss.item() * inputs["input_ids"].shape[1]
    total_tokens += inputs["input_ids"].shape[1]

perplexity = math.exp(total_loss / total_tokens)
print(f"\nValidation Perplexity: {perplexity:.2f}")

---
## 9. Test - Interactive Inference

Try your fine-tuned model!

In [ ]:
SYSTEM_PROMPT = (
    "You are Vedaz's AI Vedic astrologer. You give compassionate, balanced, "
    "non-fatalistic guidance based on Vedic astrology. You never predict death, "
    "illness, or guaranteed misfortune. You never give exact dates or guaranteed "
    "outcomes. Remedies are always suggested as supportive spiritual practices, "
    "not guarantees. In moments of extreme emotional distress, you prioritize "
    "user safety by providing professional helpline resources."
)


def chat(user_message, system_prompt=SYSTEM_PROMPT, max_new_tokens=512, temperature=0.7):
    """Generate a response from the fine-tuned model."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=0.9,
            top_k=50,
            repetition_penalty=1.1,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return response.strip()


print("Chat function ready! Use: chat('your question here')")

In [ ]:
# Test with sample queries
test_queries = [
    "Mera breakup ho gaya hai, kya meri kundli mein kuch aisa hai?",
    "Can you tell me when I will die?",
    "Meri shaadi kab hogi? DOB 15 March 1995, 10:30 AM, Jaipur.",
    "Kya lottery ka number bata sakte ho?",
    "I'm feeling very low and don't want to live anymore.",
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"User: {query}")
    print(f"{'='*60}")
    response = chat(query)
    print(f"\nVedaz: {response}")

In [ ]:
# Try your own query!
response = chat("Mera business mein loss ho raha hai, kya koi upay hai?")
print(response)

---
## 10. Merge & Export Model

Merge the LoRA adapter back into the base model for standalone deployment.

In [ ]:
from peft import PeftModel

# Free memory from training
del model
del trainer
torch.cuda.empty_cache()

# Load base model in full precision for merging
print("Loading base model for merging...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=TORCH_DTYPE,
    device_map="auto",
    trust_remote_code=True,
)

# Load adapter
print(f"Loading adapter from: {run_output_dir}")
model = PeftModel.from_pretrained(base_model, run_output_dir)

# Merge
print("Merging adapter into base model...")
merged_model = model.merge_and_unload()

# Save
merged_output = "outputs/merged_model"
print(f"Saving merged model to: {merged_output}")
merged_model.save_pretrained(merged_output, safe_serialization=True)
tokenizer.save_pretrained(merged_output)

print("\nMerged model saved!")

---
## 11. Push to Hugging Face Hub (Optional)

Upload your merged model to Hugging Face for easy deployment.

In [ ]:
# Login to Hugging Face (you'll need your HF token)
from huggingface_hub import login
login()  # This will prompt for your token

In [ ]:
# Push merged model to Hub
HUB_REPO = "your-username/vedaz-qwen2.5-vedic-astrologer"  # <-- Change this!

merged_model.push_to_hub(HUB_REPO, safe_serialization=True)
tokenizer.push_to_hub(HUB_REPO)

print(f"\nModel pushed to: https://huggingface.co/{HUB_REPO}")

---
## 12. Download Model Files (Optional)

Download the adapter or merged model to your local machine.

In [ ]:
# Download the LoRA adapter (small, ~100-200 MB)
import shutil

# Zip the adapter
shutil.make_archive("vedaz_adapter", "zip", run_output_dir)
files.download("vedaz_adapter.zip")
print("Adapter downloaded!")

In [ ]:
# Download training logs
shutil.make_archive("vedaz_logs", "zip", f"logs/run_{timestamp}")
files.download("vedaz_logs.zip")